kernal: scanpy

# Set up

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as mpdf
from matplotlib.pyplot import rc_context

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import scanpy as sc
import muon as mu

import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.filterwarnings(action='once')
warnings.simplefilter(action='once')
warnings.simplefilter(action="ignore", category=NumbaDeprecationWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)

In [ ]:
sc.settings.verbosity = 0  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(8, 7), facecolor="white")
sc.logging.print_versions()

# Load data

In [ ]:
dataset = "FL_wnn"
new_file, old_file = "v00", "v00"
new_anno = "anno_wnn_v51"

All blood

In [ ]:
data_dir = '/work/DevM_analysis/01.annotation/11.subclustering/blood/data'
mdata = mu.read(f"{data_dir}/{dataset}_clustered.v00.h5mu")
mdata

HSC

In [ ]:
mdata_hsc = mu.read(f"data/{dataset}_clustered.v01.h5mu")
mdata_hsc

Merge with HSC sub

In [ ]:
df_hsc = pd.read_csv('data/FL_wnn_cellmeta.v01.csv', index_col=0)
df_hsc = df_hsc[['leiden_wnn_0.3']].rename(columns={'leiden_wnn_0.3': 'anno_hsc_tmp'})
df_hsc['anno_hsc_tmp'] = df_hsc['anno_hsc_tmp'].astype(str).str.replace("^", 'HSC-', regex=True)
df_hsc

In [ ]:
rna = mdata['rna']
rna.obs = rna.obs.merge(df_hsc, left_index=True, right_index=True, how='left')
rna.obs['anno_hsc_tmp'] = rna.obs['anno_hsc_tmp'].fillna(rna.obs[new_anno])
rna.obs['anno_hsc_tmp'] = rna.obs['anno_hsc_tmp'].astype("category")
rna.obs['anno_hsc_tmp'] = rna.obs['anno_hsc_tmp'].cat.reorder_categories(["HSC-0", "HSC-1", "HSC-2", "HSC-3", "HSC-4", "GP", "Granulocyte",
                                                                  "MEMP-t", "MEMP", "MEP", "MEMP-Mast-Ery", "MEMP-Ery", "Early-Ery", "Late-Ery",
                                                                  "MEMP-MK", "MK", "MastP-t", "MastP", "Mast",
                                                                  "MDP", "Monocyte", "Kupffer", "cDC1", "cDC2", "pDC", "ASDC",
                                                                  "LMPP", "LP", "Cycling-LP", "PreProB", "ProB-1", "ProB-2", "Large-PreB", "Small-PreB", "IM-B",
                                                                  "NK", "ILCP", "T"])

In [ ]:
rna_hsc = mdata_hsc['rna']
rna_hsc.obs = rna_hsc.obs.merge(df_hsc, left_index=True, right_index=True, how='left')
rna_hsc.obs['anno_hsc_tmp'] = rna_hsc.obs['anno_hsc_tmp'].astype("category")
rna_hsc.obs['anno_hsc_tmp'] = rna_hsc.obs['anno_hsc_tmp'].cat.reorder_categories(["HSC-0", "HSC-1", "HSC-2", "HSC-3", "HSC-4"])

Load markers of leiden_wnn_0.3

In [ ]:
df_03 = pd.read_csv("/work/DevM_analysis/01.annotation/11.subclustering/HSC/data/FL_wnn_markerGenes.leiden_wnn_0.3.csv")

# Group by 'group', sort by 'scores' in descending order, and add "HSC-" prefix to the group column
df_03 = (
    df_03
    .sort_values(by=['group', 'scores'], ascending=[True, False])
    .assign(group=lambda x: "HSC-" + x['group'].astype(str))
)

df_03.head()

# Top 5

In [ ]:
top_genes_dict = (
    df_03.sort_values(by='scores', ascending=False)
    .groupby('group')
    .head(5)
    .groupby('group')['names']
    .apply(list)
    .to_dict()
)
top_genes_dict

HSC

In [ ]:
sc.pl.dotplot(rna_hsc, var_names=top_genes_dict, groupby=['anno_hsc_tmp'], standard_scale="var", show=False)

In all blood

In [ ]:
dp = sc.pl.dotplot(
    rna,
    var_names=top_genes_dict,
    groupby='anno_hsc_tmp',   # use string, not ['anno_hsc_tmp']
    standard_scale="var",
    figsize=(7, 6),
    return_fig=True,
    show=False,
)

dp.style(
    largest_dot=70,   # lower this more if you want smaller dots
    smallest_dot=0,
)

axes = dp.get_axes()
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)
axes["mainplot_ax"].tick_params(axis="y", labelsize=8)

dp.show()
dp.savefig("plots/HSC-subcluster-dotplot-top5.pdf", bbox_inches="tight")

# Dendrogram

RNA

In [ ]:
sc.tl.dendrogram(mdata['rna'], groupby=['anno_hsc_tmp'], n_pcs=mdata['rna'].obsm["X_harmony"].shape[1], use_rep="X_harmony",
                 cor_method="pearson", linkage_method="complete", optimal_ordering=True)

In [ ]:
with plt.rc_context({"figure.figsize": (2, len(rna.obs['anno_hsc_tmp'].cat.categories) * 0.3)}):
    sc.pl.dendrogram(mdata['rna'], groupby = 'anno_hsc_tmp', show=False, orientation = "left")
    #plt.savefig(f"{work_dir}/plots/{dataset}_dendrogramOfRNA.{new_file}.pdf", bbox_inches="tight")

ATAC

In [ ]:
mdata['atac'].obs['anno_hsc_tmp'] = rna.obs['anno_hsc_tmp'].copy()

In [ ]:
sc.tl.dendrogram(mdata['atac'], groupby=['anno_hsc_tmp'], n_pcs=mdata['atac'].obsm["X_harmony"].shape[1], use_rep="X_harmony",
                 cor_method="pearson", linkage_method="complete", optimal_ordering=True)

In [ ]:
with plt.rc_context({"figure.figsize": (2, len(rna.obs['anno_hsc_tmp'].cat.categories) * 0.3)}):
    sc.pl.dendrogram(mdata['atac'], groupby = 'anno_hsc_tmp', show=False, orientation = "left")
    #plt.savefig(f"{work_dir}/plots/{dataset}_dendrogramOfATAC.{new_file}.pdf", bbox_inches="tight")